# Graphormer + Attention CN (`graphattncn1`)

NCN variant combining **Graphormer encoding** (centrality + spatial bias) with **attention pooling** over the common-neighbour token sequence, conditioned on the target pair.

In [1]:
import sys, os, csv, gc
import torch

sys.path.insert(0, os.path.abspath('..'))

# PyTorch >= 2.6 changed weights_only default to True, breaking OGB dataset loading
_orig_load = torch.load
def _load_compat(f, *args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_load(f, *args, **kwargs)
torch.load = _load_compat

from utils.presets import make_args, CORA, CITESEER, PUBMED, COLLAB, PPA, DDI
from utils.engine  import run_experiment, get_device, free_gpu
print('imports OK')

/home/swagnik/swagnik/energy/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/swagnik/swagnik/energy/venv/lib/python3.11/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


imports OK


In [2]:
device = get_device()
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  |  VRAM: {props.total_memory/1e9:.1f} GB  |  Free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

GPU  : NVIDIA RTX 4500 Ada Generation
VRAM : 25.3 GB
PyTorch 2.10.0+cu128  |  CUDA: True
GPU: NVIDIA RTX 4500 Ada Generation  |  VRAM: 25.3 GB  |  Free: 24.8 GB


## Configuration

In [3]:
import os

PREDICTOR = 'graphattncn1'

CFG = dict(
    use_aa=True, use_ra=True, use_diff_feat=True,
    grad_clip=1.0,
    use_amp=True,
)

NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
CSV_PATH = os.path.join(NOTEBOOK_DIR, 'results.csv')

def save_result(dataset_name, result):
    if result is None:
        print(f'  [SKIP] {dataset_name} — experiment failed'); return

    # Use best test score across all epochs (not just best-val epoch)
    all_metrics = result.get('all_metrics_best_tst', result.get('all_metrics', {}))
    rows = []
    for metric in ('Hits@20', 'Hits@50', 'Hits@100'):
        if metric not in all_metrics: continue
        m = all_metrics[metric]
        rows.append({'dataset': dataset_name, 'predictor': PREDICTOR,
                     'metric': metric, 'runs': len(result.get('per_run', [])),
                     'val_mean': f"{m['val_mean']:.6f}", 'val_std': f"{m['val_std']:.6f}",
                     'tst_mean': f"{m['tst_mean']:.6f}", 'tst_std': f"{m['tst_std']:.6f}"})

    if not rows: return
    # Keep only the row with the highest test hits
    best = max(rows, key=lambda r: float(r['tst_mean']))

    existing = []
    if os.path.exists(CSV_PATH):
        with open(CSV_PATH, newline='', encoding='utf-8') as fh:
            existing = list(csv.DictReader(fh))

    # Replace stale rows for this (dataset, predictor) pair
    existing = [r for r in existing if not (r['dataset'] == dataset_name and r['predictor'] == PREDICTOR)]

    fieldnames = ['dataset','predictor','metric','runs','val_mean','val_std','tst_mean','tst_std']
    with open(CSV_PATH, 'w', newline='', encoding='utf-8') as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(existing + [best])

    print(f"  best {best['metric']}: val {best['val_mean']} ±{best['val_std']}  tst {best['tst_mean']} ±{best['tst_std']}")
    print(f'  [saved → {CSV_PATH}]')

print(f'Predictor: {PREDICTOR} | Config: {CFG}')

Predictor: graphattncn1 | Config: {'use_aa': True, 'use_ra': True, 'use_diff_feat': True, 'grad_clip': 1.0, 'use_amp': True}


## Experiments

### Cora

In [4]:
args = make_args(CORA, predictor=PREDICTOR, **CFG)
args.runs            = 1
args.report_all_hits = True
args.epochs          = 100

try:
    result = run_experiment(args)
except Exception:
    import traceback; traceback.print_exc()
    result = None
finally:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

save_result('Cora', result)

GPU  : NVIDIA RTX 4500 Ada Generation
VRAM : 25.3 GB
2708 tensor(2707)
dataset split 
train edge 3696
valid edge 527
valid edge_neg 1055
test edge 1055
test edge_neg 1055


/home/swagnik/swagnik/CNT/experiments/ogbdataset.py:31: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  data = train_test_split_edges(data, test_ratio, test_ratio)


  run 1/1  ep   1/100  loss=2.3306  Hits@100=(0.3548,0.3403)  (0.9s)
  run 1/1  ep   2/100  loss=1.3837  Hits@100=(0.6072,0.5469)  (0.2s)
  run 1/1  ep   3/100  loss=1.3516  Hits@100=(0.6262,0.5630)  (0.2s)
  run 1/1  ep   4/100  loss=1.3182  Hits@100=(0.6528,0.6275)  (0.2s)
  run 1/1  ep   5/100  loss=1.2562  Hits@100=(0.7495,0.7365)  (0.2s)
  run 1/1  ep   6/100  loss=1.1260  Hits@100=(0.8065,0.8028)  (0.2s)
  run 1/1  ep   7/100  loss=1.0664  Hits@100=(0.8235,0.8265)  (0.2s)
  run 1/1  ep   8/100  loss=0.9917  Hits@100=(0.8368,0.8701)  (0.2s)
  run 1/1  ep   9/100  loss=0.9528  Hits@100=(0.8501,0.8635)  (0.2s)
  run 1/1  ep  10/100  loss=0.9450  Hits@100=(0.8577,0.8550)  (0.2s)
  run 1/1  ep  11/100  loss=0.8966  Hits@100=(0.8558,0.8559)  (0.2s)
  run 1/1  ep  12/100  loss=0.8999  Hits@100=(0.8596,0.8616)  (0.2s)
  run 1/1  ep  13/100  loss=0.8822  Hits@100=(0.8748,0.8626)  (0.2s)
  run 1/1  ep  14/100  loss=0.8274  Hits@100=(0.8748,0.8758)  (0.2s)
  run 1/1  ep  15/100  loss=0.8322

### Citeseer

In [5]:
args = make_args(CITESEER, predictor=PREDICTOR, **CFG)
args.runs            = 1
args.report_all_hits = True
args.epochs          = 100

try:
    result = run_experiment(args)
except Exception:
    import traceback; traceback.print_exc()
    result = None
finally:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

save_result('Citeseer', result)

GPU  : NVIDIA RTX 4500 Ada Generation
VRAM : 25.3 GB


/home/swagnik/swagnik/CNT/experiments/ogbdataset.py:31: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  data = train_test_split_edges(data, test_ratio, test_ratio)


3327 tensor(3325)
dataset split 
train edge 3187
valid edge 455
valid edge_neg 910
test edge 910
test edge_neg 910
  run 1/1  ep   1/100  loss=1.6414  Hits@100=(0.5648,0.5868)  (0.4s)
  run 1/1  ep   2/100  loss=1.1320  Hits@100=(0.8747,0.8527)  (0.4s)
  run 1/1  ep   3/100  loss=0.8960  Hits@100=(0.8813,0.8736)  (0.4s)
  run 1/1  ep   4/100  loss=0.8161  Hits@100=(0.9121,0.8945)  (0.4s)
  run 1/1  ep   5/100  loss=0.7585  Hits@100=(0.9143,0.9044)  (0.4s)
  run 1/1  ep   6/100  loss=0.7690  Hits@100=(0.9165,0.8989)  (0.4s)
  run 1/1  ep   7/100  loss=0.7203  Hits@100=(0.9121,0.9022)  (0.4s)
  run 1/1  ep   8/100  loss=0.6756  Hits@100=(0.9319,0.9044)  (0.4s)
  run 1/1  ep   9/100  loss=0.6738  Hits@100=(0.9319,0.9143)  (0.4s)
  run 1/1  ep  10/100  loss=0.6498  Hits@100=(0.9253,0.9088)  (0.4s)
  run 1/1  ep  11/100  loss=0.6498  Hits@100=(0.9253,0.9121)  (0.4s)
  run 1/1  ep  12/100  loss=0.5965  Hits@100=(0.9297,0.9143)  (0.4s)
  run 1/1  ep  13/100  loss=0.5927  Hits@100=(0.9275,0.90

### Pubmed

In [6]:
args = make_args(PUBMED, predictor=PREDICTOR, **CFG)
args.runs            = 1
args.report_all_hits = True
args.epochs          = 100

try:
    result = run_experiment(args)
except Exception:
    import traceback; traceback.print_exc()
    result = None
finally:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

save_result('Pubmed', result)

GPU  : NVIDIA RTX 4500 Ada Generation
VRAM : 25.3 GB


/home/swagnik/swagnik/CNT/experiments/ogbdataset.py:31: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  data = train_test_split_edges(data, test_ratio, test_ratio)


19717 tensor(19716)
dataset split 
train edge 31028
valid edge 4432
valid edge_neg 8864
test edge 8864
test edge_neg 8864
  run 1/1  ep   1/100  loss=1.4244  Hits@100=(0.3202,0.3001)  (1.6s)
  run 1/1  ep   2/100  loss=1.1380  Hits@100=(0.3215,0.3019)  (1.5s)
  run 1/1  ep   3/100  loss=0.8623  Hits@100=(0.3308,0.3089)  (1.5s)
  run 1/1  ep   4/100  loss=0.7920  Hits@100=(0.3721,0.3396)  (1.5s)
  run 1/1  ep   5/100  loss=0.7218  Hits@100=(0.3867,0.3885)  (1.6s)
  run 1/1  ep   6/100  loss=0.6451  Hits@100=(0.4059,0.4146)  (1.5s)
  run 1/1  ep   7/100  loss=0.6145  Hits@100=(0.4174,0.4282)  (1.6s)
  run 1/1  ep   8/100  loss=0.5762  Hits@100=(0.4280,0.4334)  (1.7s)
  run 1/1  ep   9/100  loss=0.5208  Hits@100=(0.4571,0.4975)  (1.6s)
  run 1/1  ep  10/100  loss=0.4825  Hits@100=(0.5223,0.5240)  (1.6s)
  run 1/1  ep  11/100  loss=0.4482  Hits@100=(0.5697,0.5600)  (1.6s)
  run 1/1  ep  12/100  loss=0.4236  Hits@100=(0.6038,0.5989)  (1.6s)
  run 1/1  ep  13/100  loss=0.4058  Hits@100=(0.61

### collab

In [ ]:
args = make_args(COLLAB, predictor=PREDICTOR, **CFG)
args.runs            = 1
args.report_all_hits = True
args.epochs          = 50
args.batch_size = min(args.batch_size, 8192)
args.testbs     = min(args.testbs, 16384)

try:
    result = run_experiment(args)
except Exception:
    import traceback; traceback.print_exc()
    result = None
finally:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

save_result('collab', result)

### ppa

In [ ]:
args = make_args(PPA, predictor=PREDICTOR, **CFG)
args.runs            = 1
args.report_all_hits = True
args.epochs          = 50

try:
    result = run_experiment(args)
except Exception:
    import traceback; traceback.print_exc()
    result = None
finally:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

save_result('ppa', result)

### ddi

In [ ]:
args = make_args(DDI, predictor=PREDICTOR, **CFG)
args.runs            = 1
args.report_all_hits = True
args.epochs          = 100
# DDI is large and dense — cap batch sizes and CN sampling to avoid OOM
args.batch_size = min(args.batch_size, 256)
args.testbs     = min(args.testbs, 4096)
args.cndeg      = 64
args.trndeg     = 64
args.tstdeg     = 256

try:
    result = run_experiment(args)
except Exception:
    import traceback; traceback.print_exc()
    result = None
finally:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

save_result('ddi', result)